# 🧪 Tuần 3: Confounding Demo & Simpson's Paradox

**Mục tiêu:**
1. Tự sinh dữ liệu (Synthetic Data) với `User_ID`, đóng vai trò là "Chúa trời" để biết trước True Causal Effect (Sức mạnh thật của Voucher).
2. Cho thấy sự nguy hiểm của **Biến nhiễu (Confounder)** khi dùng Naive Estimate.
3. Chứng minh sức mạnh của **Randomization (A/B Test)** trong việc loại bỏ nhiễu.
4. Mô phỏng **Nghịch lý Simpson** bằng cách cắt lớp (Segmentation).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8-whitegrid')
np.random.seed(42) # Set seed for reproducibility

## 1. Data Generation Process (DGP) - Giả lập Vũ trụ

Chúng ta sẽ tạo ra 10,000 khách hàng. 
- **Confounder (Z):** `customer_type`. 20% là VIP, 80% là Regular.
- **Base Rides (Y0):** Khách VIP mặc định đi trung bình 10 chuyến. Khách Regular mặc định đi 4 chuyến.
- **True Treatment Effect (ATE):** Voucher thực sự làm tăng **+2 chuyến** cho bất kỳ ai nhận được.

In [ ]:
N = 10000
true_ate = 2.0

# 1. Sinh biến nhiễu Z (VIP = 1, Regular = 0)
is_vip = np.random.binomial(n=1, p=0.2, size=N)

# 2. Sinh số chuyến đi cơ bản (Base Rides Y0)
# VIP có Base ~ Normal(10, 2), Regular có Base ~ Normal(4, 1.5)
base_rides = np.where(is_vip == 1, 
                      np.random.normal(10, 2, N), 
                      np.random.normal(4, 1.5, N))

df = pd.DataFrame({
    'user_id': range(1, N + 1),
    'is_vip': is_vip,
    'base_rides': base_rides
})

df.head()

## 2. Kịch bản 1: Observational Data (Dữ liệu bị nhiễu)

Giả sử thuật toán gửi Push Notification bị thiên vị: Khách VIP online nhiều nên dễ nhận Voucher hơn.
- Khách VIP có **80%** cơ hội nhận Voucher (T=1).
- Khách Regular chỉ có **20%** cơ hội nhận Voucher (T=1).

In [ ]:
# Xác suất nhận Treatment phụ thuộc vào Z (Confounder)
prob_T_obs = np.where(df['is_vip'] == 1, 0.8, 0.2)
df['T_obs'] = np.random.binomial(n=1, p=prob_T_obs)

# Outcome = Base Rides + (ATE * Treatment) + Noise
df['Y_obs'] = df['base_rides'] + (true_ate * df['T_obs']) + np.random.normal(0, 0.5, N)
df['Y_obs'] = df['Y_obs'].clip(lower=0).round().astype(int)

print("Phân bổ Treatment trong Observational Data:")
display(pd.crosstab(df['is_vip'], df['T_obs'], normalize='index', rownames=['is_vip'], colnames=['Treatment']).round(2))

In [ ]:
# Tính Naive Estimate
mean_Y_treated_obs = df[df['T_obs'] == 1]['Y_obs'].mean()
mean_Y_control_obs = df[df['T_obs'] == 0]['Y_obs'].mean()
naive_estimate = mean_Y_treated_obs - mean_Y_control_obs

print(f"Naive Estimate (T=1 vs T=0): {naive_estimate:.2f} chuyến")
print(f"True ATE (Đáp án thật)     : {true_ate:.2f} chuyến")
print(f"=> BIAS (Độ lệch)          : {naive_estimate - true_ate:.2f} chuyến")

> **💡 Phân tích Bias:**
> Naive Estimate ra tận **~6.8 chuyến** thay vì 2.0 chuyến. Đây là **Positive Bias** (Thiên lệch dương).
> Lý do: Nhóm T=1 có chứa quá nhiều Khách VIP (những người vốn dĩ đã đi 10 chuyến/tháng). Thuật toán ngây thơ đã gán toàn bộ số chuyến đi tự nhiên của khách VIP thành công trạng của Voucher.

## 3. Nghịch lý Simpson (Simpson's Paradox)

Mặc dù gộp chung thì Naive Estimate bị sai lệch (6.8 chuyến). Nhưng nếu chúng ta chẻ nhỏ dữ liệu (Segmentation) theo từng nhóm `is_vip`, điều kỳ diệu của Causal Inference sẽ xuất hiện.

In [ ]:
# Xét riêng nhóm VIP
vip_df = df[df['is_vip'] == 1]
vip_effect = vip_df[vip_df['T_obs'] == 1]['Y_obs'].mean() - vip_df[vip_df['T_obs'] == 0]['Y_obs'].mean()

# Xét riêng nhóm Regular
reg_df = df[df['is_vip'] == 0]
reg_effect = reg_df[reg_df['T_obs'] == 1]['Y_obs'].mean() - reg_df[reg_df['T_obs'] == 0]['Y_obs'].mean()

print(f"Effect xét chung toàn tập : {naive_estimate:.2f} chuyến")
print(f"Effect trong nhóm VIP     : {vip_effect:.2f} chuyến")
print(f"Effect trong nhóm Regular : {reg_effect:.2f} chuyến")

> **💡 Nhận xét (Nghịch lý Simpson):**
> Dù hiệu ứng tổng thể (Overall) là +6.8, nhưng hiệu ứng trong *từng nhóm nhỏ* lại bằng đúng +2.0 (bằng True ATE)!
> Đây là nền tảng của các thuật toán Causal Inference (như Stratification / Matching): Khi ta cố định được Confounder (Z), ta sẽ tìm được Causal Effect (T -> Y).

## 4. Kịch bản 2: Randomized Data (Sức mạnh của A/B Testing)

Bây giờ, thay vì để thuật toán chọn, chúng ta Tung Đồng Xu 50/50 cho tất cả mọi người.

In [ ]:
# Xác suất nhận Treatment KHÔNG phụ thuộc vào Z
df['T_rand'] = np.random.binomial(n=1, p=0.5, size=N)

# Outcome tương tự
df['Y_rand'] = df['base_rides'] + (true_ate * df['T_rand']) + np.random.normal(0, 0.5, N)
df['Y_rand'] = df['Y_rand'].clip(lower=0).round().astype(int)

mean_Y_treated_rand = df[df['T_rand'] == 1]['Y_rand'].mean()
mean_Y_control_rand = df[df['T_rand'] == 0]['Y_rand'].mean()
randomized_estimate = mean_Y_treated_rand - mean_Y_control_rand

print(f"Randomized Estimate (A/B Test) : {randomized_estimate:.2f} chuyến")
print(f"True ATE (Đáp án thật)         : {true_ate:.2f} chuyến")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.barplot(data=df, x='T_obs', y='Y_obs', hue='is_vip', ax=axes[0], palette='Set2')
axes[0].set_title(f'Observational Data (Bị nhiễu)\nNaive Estimate = {naive_estimate:.2f}')
axes[0].set_ylabel('Số chuyến đi trung bình')
axes[0].set_xlabel('Treatment (Có Voucher)')

sns.barplot(data=df, x='T_rand', y='Y_rand', hue='is_vip', ax=axes[1], palette='Set2')
axes[1].set_title(f'Randomized Data (A/B Test)\nRandomized Estimate = {randomized_estimate:.2f}')
axes[1].set_ylabel('')
axes[1].set_xlabel('Treatment (Có Voucher)')

plt.tight_layout()
plt.show()

> **💡 Tổng kết Tuần 3:**
> - **Randomization (A/B Testing)** là "tiêu chuẩn vàng" vì nó phá vỡ mối liên kết giữa Biến nhiễu (VIP) và Biến can thiệp (Voucher).
> - Khi không thể chạy A/B Test (như ở Kịch bản 1), ta bắt buộc phải dùng **Causal Inference (Matching, Stratification, IPW)** để tìm ra +2.0 từ con số bị nhiễu +6.8.